# Market Basket Analysis

Finds association rules between **product categories** purchased within the same order.

**Metrics reported**
- **Support** – fraction of orders that contain the itemset
- **Confidence** – P(consequent | antecedent)
- **Lift** – how much more likely the consequent is given the antecedent vs. by chance

Results are sorted by **confidence (descending)**.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from itertools import combinations

In [2]:
# ── Load data ──────────────────────────────────────────────────────────────────
orders      = pd.read_parquet("../EDA/outputs_finals/orders.parquet")
lines       = pd.read_parquet("../EDA/outputs_finals/lines.parquet")
cust        = pd.read_parquet("../EDA/outputs_finals/customers.parquet")
rc_checkout = pd.read_parquet("../EDA/outputs_finals/rc_checkout.parquet")
rc_recurring= pd.read_parquet("../EDA/outputs_finals/rc_recurring.parquet")

orders["order_date"] = pd.to_datetime(orders["order_date"], utc=True)
lines["order_date"]  = pd.to_datetime(lines["order_date"],  utc=True)

print(f"Orders: {orders.shape[0]:,} rows")
print(f"Lines:  {lines.shape[0]:,} rows")

Orders: 9,852 rows
Lines:  16,040 rows


In [3]:
# ── Build basket: one row per order, one column per product category ────────────
# Each cell = 1 if that category appeared in the order, else 0

basket = (
    lines
    .groupby(["order_id", "product_category"])["Line: Quantity"]
    .sum()
    .unstack(fill_value=0)
    .reset_index(drop=True)
)

# Binarise: any quantity > 0 → True
basket_bool = basket.map(lambda x: True if x > 0 else False)

print(f"Basket shape: {basket_bool.shape}")
print(f"Product categories ({basket_bool.shape[1]}):")
print(list(basket_bool.columns))

Basket shape: (9852, 7)
Product categories (7):
['Accessories', 'Clear Protein', 'Collagen Glow', 'Lean Protein', 'Other', 'Soy Protein', 'Unknown']


In [4]:
# ── Apriori — frequent itemsets ────────────────────────────────────────────────
# min_support = 0.01 (1% of orders).  Lower if you have few categories.

MIN_SUPPORT = 0.01

frequent_itemsets = apriori(
    basket_bool,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=None,          # no cap on itemset size
)

frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)

print(f"Frequent itemsets found: {len(frequent_itemsets):,}")
frequent_itemsets.sort_values("support", ascending=False).head(20)

Frequent itemsets found: 20


,support,itemsets,length
4,0.352314,(Other),1
6,0.240966,(Unknown),1
1,0.233760,(Clear Protein),1
3,0.218433,(Lean Protein),1
0,0.117032,(Accessories),1
2,0.098457,(Collagen Glow),1
5,0.050142,(Soy Protein),1
8,0.040804,"(Lean Protein, Accessories)",2
12,0.039180,"(Clear Protein, Lean Protein)",2
7,0.036033,"(Clear Protein, Accessories)",2


In [5]:
# ── Association rules — ALL pairs, sorted by confidence desc ──────────────────

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.0,     # keep every rule; filter below if needed
    num_itemsets=len(frequent_itemsets),
)

# Pretty-print antecedent / consequent as strings
rules["antecedents"] = rules["antecedents"].apply(lambda x: " + ".join(sorted(x)))
rules["consequents"] = rules["consequents"].apply(lambda x: " + ".join(sorted(x)))

# Keep only the metrics we care about and sort by confidence
rules_out = (
    rules[[
        "antecedents", "consequents",
        "support", "confidence", "lift",
        "leverage", "conviction", "zhangs_metric",
    ]]
    .sort_values("confidence", ascending=False)
    .reset_index(drop=True)
)

print(f"Total rules generated: {len(rules_out):,}")
rules_out.head(20)

Total rules generated: 26


,antecedents,consequents,support,confidence,lift,leverage,conviction,zhangs_metric
0,Accessories,Lean Protein,0.040804,0.348656,1.596169,0.015240,1.199929,0.423005
1,Accessories,Clear Protein,0.036033,0.307892,1.317133,0.008676,1.107112,0.272688
2,Accessories,Other,0.032176,0.274935,0.780369,-0.009056,0.893280,-0.241706
3,Collagen Glow,Other,0.025782,0.261856,0.743245,-0.008906,0.877451,-0.277028
4,Soy Protein,Other,0.012688,0.253036,0.718212,-0.004978,0.867091,-0.292315
5,Accessories,Unknown,0.022026,0.188205,0.781042,-0.006175,0.935006,-0.240986
6,Lean Protein,Accessories,0.040804,0.186803,1.596169,0.015240,1.085798,0.477886
7,Lean Protein,Clear Protein,0.039180,0.179368,0.767318,-0.011881,0.933720,-0.279534
8,Clear Protein,Lean Protein,0.039180,0.167607,0.767318,-0.011881,0.938941,-0.283540
9,Clear Protein,Accessories,0.036033,0.154147,1.317133,0.008676,1.043878,0.314229


In [6]:
# ── Filter: PAIR rules only (1 antecedent → 1 consequent) ─────────────────────

pair_rules = rules_out[
    rules_out["antecedents"].str.count(r" \+ ").eq(0) &   # no " + " = single item
    rules_out["consequents"].str.count(r" \+ ").eq(0)
].copy()

print(f"Pair rules (1→1): {len(pair_rules):,}")
pair_rules

Pair rules (1→1): 26


,antecedents,consequents,support,confidence,lift,leverage,conviction,zhangs_metric
0,Accessories,Lean Protein,0.040804,0.348656,1.596169,0.015240,1.199929,0.423005
1,Accessories,Clear Protein,0.036033,0.307892,1.317133,0.008676,1.107112,0.272688
2,Accessories,Other,0.032176,0.274935,0.780369,-0.009056,0.893280,-0.241706
3,Collagen Glow,Other,0.025782,0.261856,0.743245,-0.008906,0.877451,-0.277028
4,Soy Protein,Other,0.012688,0.253036,0.718212,-0.004978,0.867091,-0.292315
5,Accessories,Unknown,0.022026,0.188205,0.781042,-0.006175,0.935006,-0.240986
6,Lean Protein,Accessories,0.040804,0.186803,1.596169,0.015240,1.085798,0.477886
7,Lean Protein,Clear Protein,0.039180,0.179368,0.767318,-0.011881,0.933720,-0.279534
8,Clear Protein,Lean Protein,0.039180,0.167607,0.767318,-0.011881,0.938941,-0.283540
9,Clear Protein,Accessories,0.036033,0.154147,1.317133,0.008676,1.043878,0.314229


In [7]:
# ── Manual pair computation (exhaustive, no support floor) ────────────────────
# This section computes ALL category pairs directly from the basket,
# bypassing any min_support filter so you never miss a low-frequency pair.

n_orders = len(basket_bool)
cats     = list(basket_bool.columns)

rows = []
for a, b in combinations(cats, 2):
    sup_a  = basket_bool[a].sum() / n_orders
    sup_b  = basket_bool[b].sum() / n_orders
    sup_ab = (basket_bool[a] & basket_bool[b]).sum() / n_orders

    if sup_a == 0 or sup_b == 0 or sup_ab == 0:
        continue

    conf_ab = sup_ab / sup_a          # P(B|A)
    conf_ba = sup_ab / sup_b          # P(A|B)
    lift    = sup_ab / (sup_a * sup_b)

    rows.append({
        "antecedent":  a, "consequent":  b,
        "support":     round(sup_ab, 6),
        "confidence":  round(conf_ab, 6),
        "lift":        round(lift, 4),
    })
    rows.append({
        "antecedent":  b, "consequent":  a,
        "support":     round(sup_ab, 6),
        "confidence":  round(conf_ba, 6),
        "lift":        round(lift, 4),
    })

all_pairs = (
    pd.DataFrame(rows)
    .sort_values("confidence", ascending=False)
    .reset_index(drop=True)
)

print(f"All directional pairs: {len(all_pairs):,}")
all_pairs

All directional pairs: 42


,antecedent,consequent,support,confidence,lift
0,Accessories,Lean Protein,0.040804,0.348656,1.5962
1,Accessories,Clear Protein,0.036033,0.307892,1.3171
2,Accessories,Other,0.032176,0.274935,0.7804
3,Collagen Glow,Other,0.025782,0.261856,0.7432
4,Soy Protein,Other,0.012688,0.253036,0.7182
5,Accessories,Unknown,0.022026,0.188205,0.7810
6,Lean Protein,Accessories,0.040804,0.186803,1.5962
7,Lean Protein,Clear Protein,0.039180,0.179368,0.7673
8,Clear Protein,Lean Protein,0.039180,0.167607,0.7673
9,Clear Protein,Accessories,0.036033,0.154147,1.3171


In [8]:
# ── Quick summary: top rules by lift (positive associations) ──────────────────

print("=== Top 10 pairs by LIFT (strongest positive associations) ===")
display(
    all_pairs.drop_duplicates(subset=["support"])  # unique pairs (undirected)
    .sort_values("lift", ascending=False)
    .head(10)
    [["antecedent", "consequent", "support", "confidence", "lift"]]
    .reset_index(drop=True)
)

print("\n=== Top 10 pairs by CONFIDENCE (most predictable) ===")
display(
    all_pairs.head(10)
    [["antecedent", "consequent", "support", "confidence", "lift"]]
)

print("\n=== Top 10 pairs by SUPPORT (most common co-purchases) ===")
display(
    all_pairs.drop_duplicates(subset=["support"])
    .sort_values("support", ascending=False)
    .head(10)
    [["antecedent", "consequent", "support", "confidence", "lift"]]
    .reset_index(drop=True)
)

=== Top 10 pairs by LIFT (strongest positive associations) ===


,antecedent,consequent,support,confidence,lift
0,Accessories,Lean Protein,0.040804,0.348656,1.5962
1,Accessories,Clear Protein,0.036033,0.307892,1.3171
2,Soy Protein,Collagen Glow,0.003959,0.078947,0.8018
3,Accessories,Unknown,0.022026,0.188205,0.7810
4,Accessories,Other,0.032176,0.274935,0.7804
5,Lean Protein,Clear Protein,0.039180,0.179368,0.7673
6,Collagen Glow,Other,0.025782,0.261856,0.7432
7,Collagen Glow,Accessories,0.008323,0.084536,0.7223
8,Soy Protein,Other,0.012688,0.253036,0.7182
9,Soy Protein,Accessories,0.003553,0.070850,0.6054



=== Top 10 pairs by CONFIDENCE (most predictable) ===


,antecedent,consequent,support,confidence,lift
0,Accessories,Lean Protein,0.040804,0.348656,1.5962
1,Accessories,Clear Protein,0.036033,0.307892,1.3171
2,Accessories,Other,0.032176,0.274935,0.7804
3,Collagen Glow,Other,0.025782,0.261856,0.7432
4,Soy Protein,Other,0.012688,0.253036,0.7182
5,Accessories,Unknown,0.022026,0.188205,0.7810
6,Lean Protein,Accessories,0.040804,0.186803,1.5962
7,Lean Protein,Clear Protein,0.039180,0.179368,0.7673
8,Clear Protein,Lean Protein,0.039180,0.167607,0.7673
9,Clear Protein,Accessories,0.036033,0.154147,1.3171



=== Top 10 pairs by SUPPORT (most common co-purchases) ===


,antecedent,consequent,support,confidence,lift
0,Accessories,Lean Protein,0.040804,0.348656,1.5962
1,Lean Protein,Clear Protein,0.039180,0.179368,0.7673
2,Accessories,Clear Protein,0.036033,0.307892,1.3171
3,Unknown,Other,0.033293,0.138163,0.3922
4,Accessories,Other,0.032176,0.274935,0.7804
5,Lean Protein,Other,0.031466,0.144052,0.4089
6,Collagen Glow,Other,0.025782,0.261856,0.7432
7,Accessories,Unknown,0.022026,0.188205,0.7810
8,Soy Protein,Other,0.012688,0.253036,0.7182
9,Collagen Glow,Clear Protein,0.012586,0.127835,0.5469


In [9]:
# ── Export ─────────────────────────────────────────────────────────────────────
# OUTPUT_DIR = "../EDA/outputs"   # adjust to your OUTPUT_DIR if needed

# rules_out.to_csv(f"{OUTPUT_DIR}/05_mba_all_rules.csv",  index=False)
# all_pairs.to_csv(f"{OUTPUT_DIR}/05_mba_all_pairs.csv",  index=False)
# pair_rules.to_csv(f"{OUTPUT_DIR}/05_mba_pair_rules.csv", index=False)

# print("Saved:")
# print(f"  05_mba_all_rules.csv  — {len(rules_out):,} rules (all lengths, apriori)")
# print(f"  05_mba_all_pairs.csv  — {len(all_pairs):,} directional pairs (manual, no support floor)")
# print(f"  05_mba_pair_rules.csv — {len(pair_rules):,} 1→1 pair rules (apriori)")
# print("\n[05] Market Basket Analysis done.")